<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026/%D0%93%D0%BB%D0%B0%D0%B2%D0%B0_4_%D0%9C%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5_%D0%BC%D0%B5%D1%82%D0%BE%D0%B4%D1%8B_k%E2%80%91%D0%B1%D0%BB%D0%B8%D0%B6%D0%B0%D0%B9%D1%88%D0%B8%D1%85_%D1%81%D0%BE%D1%81%D0%B5%D0%B4%D0%B5%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Глава 4. Метрические методы: k‑ближайших соседей

Логистическая регрессия строит разделяющую поверхность, опираясь на предположение о линейной зависимости логита от признаков. Но что, если мы не хотим делать никаких предположений о форме границы и полагаемся исключительно на близость объектов в признаковом пространстве? Метод k‑ближайших соседей — это непараметрический, интуитивно понятный алгоритм, который принимает решение, глядя на «соседей» нового объекта. Он служит естественным мостом между параметрическими линейными методами и полностью непараметрическими моделями вроде деревьев решений.

---

## 1. Идея метода: геометрия и формализация

Основной принцип метода ближайших соседей можно выразить фразой: «скажи мне, кто твои соседи, и я скажу, кто ты». Для классификации это означает, что новый объект получает метку, которую имеет большинство из $k$ его ближайших соседей в обучающей выборке; для регрессии ответом служит среднее (или взвешенное среднее) значений целевой переменной этих соседей.

Формально, пусть обучающая выборка состоит из пар $\{(x_i, y_i)\}_{i=1}^n$, где $x_i \in \mathbb{R}^D$ — вектор признаков, $y_i$ — целевая переменная (класс или вещественное число). Для нового объекта $x$ определим множество индексов его $k$ ближайших соседей $N_k(x)$ относительно заданной метрики $d$:
$$
N_k(x) = \{ i_1, i_2, \dots, i_k \} \subset \{1,\dots,n\},
$$
где индексы упорядочены по возрастанию расстояния $d(x, x_i)$. Тогда решающее правило классификации:
$$
\hat{y}(x) = \arg\max_{c} \sum_{i \in N_k(x)} \mathbb{1}(y_i = c),
$$
а для регрессии:
$$
\hat{y}(x) = \frac{1}{k} \sum_{i \in N_k(x)} y_i.
$$

Ключевую роль играет выбор метрики $d$. Чаще всего применяется евклидово расстояние $\|x - x_i\|_2 = \sqrt{\sum_{j=1}^D (x^{(j)} - x_i^{(j)})^2}$, однако допустимы и другие меры, например манхэттенское расстояние $\|x - x_i\|_1$, расстояние Минковского $\|x - x_i\|_p$, косинусное расстояние $1 - \frac{\langle x, x_i\rangle}{\|x\|\|x_i\|}$ для текстовых данных, а также специальные метрики для категориальных признаков. Выбор метрики определяет форму окрестности: евклидова метрика даёт сферические окрестности, манхэттенская — ромбовидные, а косинусная — конусообразные. На практике метрику подбирают исходя из природы данных.

---

## 2. Выбор гиперпараметров

Главным гиперпараметром является **число соседей $k$**. При $k = 1$ граница решений проходит максимально близко к точкам обучающей выборки — модель имеет очень низкое смещение, но крайне высокую дисперсию: малейшее изменение состава обучения приводит к резкому изменению границы (переобучение). В противоположном крайнем случае $k = n$ предсказание всегда равно глобальному среднему (или наиболее частому классу) — смещение максимально, дисперсия нулевая (недообучение).

Формально связь с bias‑variance tradeoff можно проиллюстрировать для регрессии. Предположим, что в достаточно малой окрестности точки $x$ истинная функция отклика меняется слабо, а ошибки наблюдений некоррелированы с дисперсией $\sigma^2$. Тогда предсказание как среднее по $k$ соседям имеет дисперсию примерно $\sigma^2/k$ (при независимых ошибках), то есть с ростом $k$ дисперсия убывает. Однако при этом в окрестность попадают всё более удалённые точки, и локальное среднее перестаёт правильно отражать условное среднее в точке $x$ — растёт смещение. Оптимальное $k$ находят компромиссом, чаще всего с помощью кросс‑валидации.

Второй гиперпараметр — **метрика расстояния**. Помимо перечисленных выше, возможно использование метрики Минковского с параметром $p$; частные случаи $p=1$ (L1), $p=2$ (L2), $p\to\infty$ (равномерная норма). Важно помнить, что с ростом размерности пространства все точки становятся почти равноудалёнными (концентрация меры), поэтому различие между метриками размывается. Для текстов, представленных в виде мешка слов, часто используют косинусное расстояние. Для гетерогенных признаков можно применять взвешенные расстояния, например расстояние Говера.

Третий важный инструмент — **взвешенные соседи**. Вместо простого усреднения можно назначать веса, обратно пропорциональные расстоянию до точки:
$$
\hat{y}(x) = \frac{\sum_{i \in N_k(x)} w_i y_i}{\sum_{i \in N_k(x)} w_i}, \quad w_i = \frac{1}{d(x, x_i) + \varepsilon},
$$
где $\varepsilon$ — малая константа для предотвращения деления на ноль. Взвешивание делает предсказание более гладким и уменьшает влияние далёких соседей внутри $k$-окрестности, что особенно полезно при неравномерной плотности обучающих данных.

---

## 3. Проклятие размерности

Интуитивно проклятие размерности проявляется в том, что с увеличением размерности $D$ объём пространства растёт экспоненциально. Точки, равномерно распределённые в единичном гиперкубе, оказываются в среднем всё дальше друг от друга. Например, среднее расстояние между двумя случайными точками в $D$-мерном единичном гиперкубе при евклидовой метрике асимптотически равно $\sqrt{D/6}$ при $D \to \infty$. Чтобы сохранить фиксированную плотность покрытия, требуется экспоненциально больше данных.

Для метода k‑NN это означает, что понятие «близости» размывается: окрестность, содержащая $k$ ближайших соседей, становится огромной по сравнению с масштабом изменения целевой переменной. Локальная аппроксимация теряет смысл, и качество модели падает. На практике k‑NN обычно хорошо работает при $D \lesssim 20$, но для более высоких размерностей требуется либо предварительное снижение размерности (PCA), либо использование альтернативных методов.

**Углублённый взгляд: асимптотическая состоятельность k‑NN при $n \to \infty$.**  
Классический результат (Stone, 1977) утверждает, что если $k \to \infty$ и $k/n \to 0$ при $n \to \infty$, то оценка регрессии методом k‑NN является состоятельной оценкой условного среднего $\mathbb{E}[Y \mid X=x]$ для почти всех $x$ относительно распределения $X$. Аналогичное утверждение верно для классификации: риск k‑NN сходится к байесовскому риску. Однако скорость сходимости катастрофически зависит от размерности: для гладких функций регрессии она составляет $O(n^{-2/(2+D)})$. Это делает метод крайне требовательным к объёму данных при больших $D$.

---

## 4. Алгоритмические аспекты

Наивная реализация поиска ближайших соседей для каждого запроса перебирает все $n$ точек и вычисляет расстояния, что имеет сложность $O(D n)$ на один запрос и $O(D n^2)$ для предсказания на всём тестовом множестве — неприемлемо для больших выборок. Чтобы ускорить поиск, используют специальные пространственные структуры данных.

**KD‑дерево** (k‑dimensional tree) рекурсивно разбивает пространство гиперплоскостями, параллельными координатным осям. В каждом узле выбирается одна из размерностей и точка разбиения (обычно медиана значений признака), после чего точки разделяются на левое и правое поддеревья. При поиске $k$ ближайших соседей для запроса мы обходим дерево, спускаясь к листу, а затем поднимаемся, проверяя, может ли другой подузел содержать более близкие точки. Если расстояние от запроса до границы узла превышает текущий радиус поиска, весь узел можно отбросить без вычисления расстояний до всех его точек. Средняя сложность поиска — $O(\log n)$ для низких размерностей, но при $D$, сравнимых с $\log n$, KD‑дерево вырождается до почти полного перебора.

**Ball‑дерево** использует иерархию вложенных гиперсфер. Каждый узел хранит центр и радиус минимальной сферы, покрывающей все точки поддерева. При поиске отсечение происходит, если расстояние от запроса до сферы узла больше текущего радиуса. Ball‑деревья обычно работают лучше KD‑деревьев в высоких размерностях.

Псевдокод алгоритма поиска ближайшего соседа в KD‑дереве выглядит так:

```
def search(node, query, best):
    если node — лист:
        обновить best расстоянием до точки
        вернуть
    определить, в какой дочерний узел спускаться первым
    search(ближайший_дочерний, query, best)
    если расстояние до разделяющей границы другого дочернего < best:
        search(другой_дочерний, query, best)
```

---

## 5. Реализация с нуля и сравнение с библиотечной версией

Приведём реализацию k‑NN классификатора с нуля на Python. Будем использовать евклидово расстояние, единичные или взвешенные голоса.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

class SimpleKNN:
    def __init__(self, n_neighbors=5, weights='uniform'):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        # матрица расстояний (n_query x n_train)
        dist = np.linalg.norm(X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :], axis=2)
        # индексы k ближайших
        idx = np.argsort(dist, axis=1)[:, :self.n_neighbors]
        k_labels = self.y_train[idx]
        if self.weights == 'uniform':
            # большинство голосов
            pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=k_labels)
        else:
            # взвешенные веса
            dist_k = np.take_along_axis(dist, idx, axis=1)
            weights = 1.0 / (dist_k + 1e-8)
            # для каждого класса суммируем веса
            classes = np.unique(self.y_train)
            weighted_votes = np.zeros((X.shape[0], len(classes)))
            for j, c in enumerate(classes):
                mask = (k_labels == c)
                weighted_votes[:, j] = np.sum(weights * mask, axis=1)
            pred = classes[np.argmax(weighted_votes, axis=1)]
        return pred
```

Проверим на реальном датасете breast_cancer и сравним с реализацией sklearn.

```python
# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Стандартизация
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Наша реализация
my_knn = SimpleKNN(n_neighbors=5)
my_knn.fit(X_train_sc, y_train)
my_pred = my_knn.predict(X_test_sc)
print("My KNN accuracy:", accuracy_score(y_test, my_pred))

# Sklearn
sk_knn = KNeighborsClassifier(n_neighbors=5)
sk_knn.fit(X_train_sc, y_train)
sk_pred = sk_knn.predict(X_test_sc)
print("Sklearn KNN accuracy:", accuracy_score(y_test, sk_pred))
```

Результаты должны совпадать (с точностью до возможного различия в весовых схемах). Теперь визуализируем границы решений на синтетических двумерных данных (moons) для разных $k$.

```python
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

def plot_decision_boundary(model, X, y, ax, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='viridis')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, ax in zip([1, 5, 15], axes):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_m, y_train_m)
    plot_decision_boundary(knn, X_train_m, y_train_m, ax, f'k = {k}')
plt.tight_layout()
plt.show()
```

Графики покажут, что при $k=1$ граница сильно изрезана и присутствуют «островки» (высокая дисперсия), при $k=15$ граница становится более гладкой, но хуже подстраивается под структуру данных (смещение).

---

## 6. Эксперименты: влияние $k$, метрики и размерности

Продолжим исследование на breast_cancer. Построим кривую зависимости точности на обучении и тесте от $k$.

```python
train_acc, test_acc = [], []
ks = range(1, 31)
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test_sc)))

plt.figure(figsize=(8,5))
plt.plot(ks, train_acc, label='Train')
plt.plot(ks, test_acc, label='Test')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Влияние k на точность')
plt.show()
```

Типичная картина: на обучении точность максимальна при $k=1$ и снижается с ростом $k$; на тесте наблюдается пик в районе $k \approx 5-10$. Это наглядно демонстрирует компромисс смещения и дисперсии.

Сравним разные метрики:

```python
metrics = ['euclidean', 'manhattan', 'cosine']
for m in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=m)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    print(f"Metric {m}: accuracy = {acc:.4f}")
```

Для низкоразмерных данных с числовыми признаками евклидова и манхэттенская метрики дают схожие результаты; косинусная может работать хуже, если не применяется специальная нормировка.

Важность стандартизации легко увидеть, сравнив точность на нестандартизованных и стандартизованных данных:

```python
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)               # без стандартизации
acc_raw = accuracy_score(y_test, knn.predict(X_test))
knn.fit(X_train_sc, y_train)
acc_sc = accuracy_score(y_test, knn.predict(X_test_sc))
print(f"Без стандартизации: {acc_raw:.4f}, со стандартизацией: {acc_sc:.4f}")
```

Без стандартизации точность значительно падает, так как признаки с большим масштабом подавляют вклад остальных в расстояние.

Для демонстрации проклятия размерности сгенерируем синтетические данные с двумя гауссовскими классами в пространствах разной размерности:

```python
from sklearn.datasets import make_classification

dims = [2, 10, 50, 100, 200]
accs = []
for D in dims:
    X_s, y_s = make_classification(n_samples=500, n_features=D, n_informative=D, n_redundant=0,
                                   n_clusters_per_class=1, random_state=42)
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.3, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_s)
    X_test_s = scaler.transform(X_test_s)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_s, y_train_s)
    acc = accuracy_score(y_test_s, knn.predict(X_test_s))
    accs.append(acc)
    print(f"D={D}: accuracy={acc:.3f}")

plt.figure(figsize=(8,5))
plt.plot(dims, accs, 'o-')
plt.xlabel('Число признаков D')
plt.ylabel('Accuracy')
plt.title('Проклятие размерности для k‑NN')
plt.grid(alpha=0.3)
plt.show()
```

С ростом $D$ точность k‑NN монотонно падает: при $D=200$ она часто приближается к случайному угадыванию.

---

## 7. Связь с байесовским подходом и другими методами

Метод k‑NN можно вывести как простейшую непараметрическую оценку плотности. Пусть в окрестности объёма $V$ точки $x$ находится $k$ объектов. Тогда оценка плотности в точке $x$ равна $\hat{p}(x) = \frac{k}{n V}$. Отношение числа соседей разных классов даёт оценку отношения условных плотностей, а решение по большинству голосов аппроксимирует байесовское решающее правило.

**Углублённый взгляд: сходимость к байесовскому классификатору.**  
При $k \to \infty$ и $k/n \to 0$ (при $n \to \infty$) оценка плотности сходится к истинной в каждой точке, следовательно, и классификатор k‑NN стремится к оптимальному байесовскому классификатору. Строгое доказательство основано на законе больших чисел и свойствах локального полиномиального сглаживания. Этот результат важен теоретически, но на практике ограничен проклятием размерности.

По сравнению с логистической регрессией и SVM метод k‑NN является полностью непараметрическим: он не строит явной глобальной модели и не оценивает параметры. Это делает его чрезвычайно гибким, но одновременно лишает интерпретируемости: нельзя сказать, какой признак важен, а граница решений существует лишь в виде набора точек. При этом k‑NN часто служит отличным «бенчмарком»: если более сложная модель не превосходит k‑NN, её применение вряд ли оправдано.

---

## Вопросы для самопроверки

### Для начинающих
1. Как работает k‑NN для классификации и регрессии?
2. Что происходит с границей решений при $k=1$ и при $k=n$?
3. Какие метрики расстояния вы знаете и в каких ситуациях они применяются?
4. Почему перед применением k‑NN необходимо стандартизировать признаки?
5. В чём проявляется проклятие размерности для k‑NN?
6. Как выбрать число соседей $k$ на практике?

### Для профессионалов
1. Докажите, что k‑NN является состоятельной оценкой условного среднего при $k\to\infty$ и $k/n\to0$.
2. Почему KD‑дерево деградирует при большой размерности? Дайте формальное обоснование.
3. Выведите оптимальную весовую схему для регрессии, минимизирующую локальную среднеквадратичную ошибку.
4. Сравните k‑NN с ядерной оценкой Надарая–Ватсона: в чём сходство и различие?
5. Как k‑NN можно интерпретировать как оценку отношения плотностей для классификации?

## Задания повышенной сложности
1. Реализуйте KD‑дерево для поиска k ближайших соседей и сравните время работы с наивным перебором при различных $n$ и $D$.
2. Докажите, что среднее евклидово расстояние между двумя равномерно распределёнными точками в $D$-мерном единичном гиперкубе асимптотически равно $\sqrt{D/6}$.
3. Постройте кривые обучения для k‑NN и логистической регрессии на breast_cancer; проанализируйте, как размер выборки влияет на смещение и дисперсию каждого метода.
4. Реализуйте взвешенную k‑NN регрессию и сравните её с обычной на данных с неравномерной плотностью признаков.
5. Докажите, что в пределе $n\to\infty, k\to\infty, k/n\to0$ байесовская ошибка классификации для k‑NN достигается (риск сходится к байесовскому риску).